# PnL Attribution

**Objective**: Comprehensive demonstration of FX hedge trade P&L attribution - decomposing total P&L into carry and spot movement components.

**P&L Attribution**

Without MtM valuation, a forward FX at maturity realizes P&L from two main sources:
```
Total P&L = Carry Component + Spot Movement Component
Sᴛ - F₀ = (S₀ - F₀) + (Sᴛ - S₀)

Where:
  • Carry Component: P&L from forward premium/discount S₀ - F₀
  • Spot Movement Component: P&L from actual FX rate change Sᴛ - S₀ 
  • F₀ = Forward rate at inception
  • S₀ = Spot rate at inception  
  • Sᴛ = Spot rate at maturity
```

**Why This Matters:**

- **Performance Reporting**: Separate hedge costs from market moves
- **Risk Management**: Understand P&L drivers

**This Notebook Structure:**

1. Setup & Load Data
2. Generate Trade History (backtesting)
3. Calculate P&L Attribution (decomposition)
4. Single Trade Example (standard & inverted pairs)
5. Aggregate P&L Analysis
6. Visualization (waterfall, time series)
7. Use Cases & Interpretation

**Contact:** kou001@e.ntu.edu.sg


## 1. Setup & Load Data

In [1]:
# import os
import sys
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
# import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Import framework modules
from src import Portfolio, ForwardCurve, load_forward_curve_config
from src import HedgeManager, HedgePosition
# from src import HedgeTrade, TradePortfolio  # NEW: Import trade objects
from src import ConstantRatioStrategy

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Dependencies loaded")

Dependencies loaded


### 1.1 Load Portfolio from NAV History

In [2]:
# Load NAV history from CSV
nav_csv_path = Path.cwd().parent / 'data' / 'raw' / 'nav_history.csv'
nav_history_df = pd.read_csv(nav_csv_path, parse_dates=['Date'], index_col='Date')
display(nav_history_df.head())

# Create portfolio
portfolio = Portfolio.from_nav_history(
    nav_df=nav_history_df,
    home_currency='USD'
)

# Get date range
START_DATE = nav_history_df.index[0]
END_DATE = nav_history_df.index[-1]

portfolio.print_info()

,EUR,GBP,JPY,USD,Total
Date,,,,,
2020-01-31,289887.7700,99973.2187,0.0000,604302.1633,994163.1520
2020-02-29,286019.6059,104200.4205,0.0000,607791.5849,998011.6113
2020-03-31,310695.0600,110965.4595,0.0000,649546.9016,1071207.4211
2020-04-30,310980.7966,106399.0882,0.0000,641338.3666,1058718.2514
2020-05-31,330957.7295,109091.4412,0.0000,654385.5641,1094434.7349



Portfolio initialized: USD-denominated
Date Range: 2020-01-31 to 2024-12-31
Frequency: Monthly (M)
IMPORTANT: All currency exposures are valued in USD
  - Each currency's NAV represents USD value of those assets
  - Returns must be in USD terms (FX effects embedded)
  - Example: EUR weight of 30% = $289,888 worth of EUR assets



### 1.2 Load Forward Curves

In [3]:
# Load forward curve configuration
config_file = Path.cwd().parent / 'data' / 'raw' / 'forward_curve_tickers.csv'
config = load_forward_curve_config(config_file)

# Load forward curves for all major pairs
currency_pairs = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCAD'] # Major currency pairs, one extra more than needed here
forward_curves = {
    pair: ForwardCurve.fetch(
        config=config,
        currency=pair,
        start_date=START_DATE,
        end_date=END_DATE + timedelta(days=100),  # Buffer period for spot rate at maturity 
        periodicity='D'
    ) for pair in currency_pairs
}

for pair, curve in forward_curves.items():
    print(f"{pair}: Frequency = {curve.periodicity}, Data Range = {curve.start_date} to {curve.end_date}")

EURUSD: Frequency = D, Data Range = 2020-01-31 00:00:00 to 2025-04-10 00:00:00
GBPUSD: Frequency = D, Data Range = 2020-01-31 00:00:00 to 2025-04-10 00:00:00
USDJPY: Frequency = D, Data Range = 2020-01-31 00:00:00 to 2025-04-10 00:00:00
USDCAD: Frequency = D, Data Range = 2020-01-31 00:00:00 to 2025-04-10 00:00:00


### 1.3 Initialize HedgeManager with ConstantRatioStrategy

We'll use `ConstantRatioStrategy` to maintain fixed hedge ratios for each currency, independent of carry considerations.

In [4]:
# Create hedge manager with constant ratio strategy
strategy = ConstantRatioStrategy(
    ratios={'EUR': 0.9, 'GBP': 0.6, 'JPY': 0.7},  
    default_tenor='3M'
)

hedge_manager = HedgeManager(
    portfolio=portfolio,
    forward_curves=forward_curves,
    strategy=strategy
)

## 2. Generate Trade History

Generate hedge trades over the full historical period. These trades will serve as the basis for P&L attribution.

**Assumptions:**
- forward hedges initiated and matures at next initialization
- Hedges held to maturity
- No early termination or rolling

In [20]:
# Generate quarterly hedge trades over the full history
# NOTE: generate_trade_history() now returns TradePortfolio object
print("="*80)
print("GENERATING TRADE HISTORY")
print("="*80)


trade_portfolio = hedge_manager.generate_trade_history(
tenor='3M',
rolling_hedges=True,
frequency='QE'  # Quarter-end, shifts to next FC date if needed
)

print(f"\nGenerated {len(trade_portfolio)} hedge trades")
print(f"Date range: {trade_portfolio.trades[0].trade_date.strftime('%Y-%m-%d')} "
      f"to {trade_portfolio.trades[-1].trade_date.strftime('%Y-%m-%d')}")

# Get unique currencies from trades
currencies = set(t.foreign_currency for t in trade_portfolio.trades)
print(f"Currencies: {', '.join(sorted(currencies))}")

# Display sample trades
print("\n" + "="*80)
print("SAMPLE TRADES")
print("="*80)

for i, trade in enumerate(trade_portfolio.trades[:]):
    print(f"\n{i+1}. {trade.trade_id}")
    print(f"   Date: {trade.trade_date.strftime('%Y-%m-%d')} | Currency: {trade.foreign_currency} | "
          f"Pair: {trade.currency_pair} | Direction: {trade.direction}")
    print(f"   Notional: ${trade.trade_notional_base:,.0f} {trade.currency_pair[:3]} | "
          f"Spot: {trade.spot_at_inception:.4f} | Forward: {trade.forward_rate:.4f}")

GENERATING TRADE HISTORY

GENERATING FREQUENCY-BASED HEDGE DATES
Frequency: QE
NAV date range: 2020-01-31 to 2024-12-31
FC date range: 2020-01-31 to 2025-04-10
Rolling period: 2020-01-31 to 2024-12-31

  Trade 1: 2020-01-31 (FC available)
  Trade 2: 2020-03-31 (FC available)
  Trade 3: 2020-06-30 (FC available)
  Trade 4: 2020-09-30 (FC available)
  Trade 5: 2020-12-31 (FC available)
  Trade 6: 2021-03-31 (FC available)
  Trade 7: 2021-06-30 (FC available)
  Trade 8: 2021-09-30 (FC available)
  Trade 9: 2021-12-31 (FC available)
  Trade 10: 2022-03-31 (FC available)
  Trade 11: 2022-06-30 (FC available)
  Trade 12: 2022-09-30 (FC available)
  Trade 13: 2023-01-02 (shifted +2d from 2022-12-31, FC available)
  Trade 14: 2023-03-31 (FC available)
  Trade 15: 2023-06-30 (FC available)
  Trade 16: 2023-10-02 (shifted +2d from 2023-09-30, FC available)
  Trade 17: 2024-01-01 (shifted +1d from 2023-12-31, FC available)
  Trade 18: 2024-04-01 (shifted +1d from 2024-03-31, FC available)
  Trade

## 3. Calculate P&L Attribution

Now we'll calculate the realized P&L for all matured trades, decomposing into carry and spot movement components. This is a simplification without consider delivery and settlement convention.

**FX Pair Quotation Convention:**

In FX markets, rates are quoted as: **Quote Currency per Base Currency**


**The P&L breakdown:**

For a forward FX, P&L calculated in the **quote currency**:

```
Total P&L = trade_sign × (Sᴛ - F₀) × trade_notional_base

Decompose as:
  Carry P&L = trade_sign × (S₀ - F₀) × trade_notional_base  
              [Premium/discount: how much you earn/pay for locking in forward]
  
  Spot P&L  = trade_sign × (Sᴛ - S₀) × trade_notional_base 
              [Spot movement: profit/loss from FX rate changes]

Where:
  trade_sign = -1 for SELL, +1 for BUY
  S₀ = Spot rate at inception (quote per base)
  F₀ = Forward rate at inception (quote per base)
  Sᴛ = Spot rate at maturity (quote per base)
  
  trade_notional_base = Notional amount in BASE CURRENCY (first currency in pair)
  Result P&L = In QUOTE CURRENCY (second currency in pair)
```



---


In [6]:
# Calculate P&L for all matured trades
# IMPORTANT: as_of_date must be AFTER trade maturity dates for complete P&L calculation as we dont have swap rate for MTM
as_of_date = trade_portfolio.trades[-1].maturity_date

print("="*80)
print("CALCULATING P&L ATTRIBUTION FOR MATURED TRADES")
print("="*80)

# Determine appropriate as_of_date for finding matured trades
# Strategy: Use the latest available forward curve date (trades that old must have matured by then)

# Get earliest and latest trade dates from the TradePortfolio
earliest_trade = trade_portfolio.trades[0].trade_date
latest_trade = trade_portfolio.trades[-1].trade_date

print(f"\nTrade History Timeline:")
print(f"  Earliest trade: {earliest_trade.strftime('%Y-%m-%d')}")
print(f"  Latest trade: {latest_trade.strftime('%Y-%m-%d')}")
print(f"  Latest data available: {as_of_date.strftime('%Y-%m-%d')}")

CALCULATING P&L ATTRIBUTION FOR MATURED TRADES

Trade History Timeline:
  Earliest trade: 2020-01-31
  Latest trade: 2024-12-31
  Latest data available: 2025-04-01


In [7]:
# Calculate P&L for all matured trades
# P&L is automatically calculated for matured trades in TradePortfolio

print("="*80)
print("ANALYZING MATURED TRADES")
print("="*80)

# Get matured trades from TradePortfolio
matured_trades = trade_portfolio.matured_trades

print(f"\nTrade Analysis as of: {as_of_date.strftime('%Y-%m-%d')}")
print(f"Total trades generated: {len(trade_portfolio)}")
print(f"Matured trades: {len(matured_trades)}")
print(f"Trades still open: {len(trade_portfolio) - len(matured_trades)}")

if len(matured_trades) > 0:
    # Get summary metrics from TradePortfolio
    print("\n" + "="*80)
    print("P&L SUMMARY")
    print("="*80)
    
    summary = trade_portfolio.summary
    print(f"\nAggregated P&L (All {summary['matured_count']} matured trades):")
    print(f"  Total Notional:         ${summary['total_notional']:>12,.0f}")
    print(f"  Total Carry P&L:        ${summary['carry_pnl']:>12,.0f}  ({summary['carry_pnl_pct']:>6.2f}%)")
    print(f"  Total Spot Movement:    ${summary['spot_pnl']:>12,.0f}  ({summary['spot_pnl_pct']:>6.2f}%)")
    print(f"  Total P&L:              ${summary['total_pnl']:>12,.0f}  ({summary['total_pnl_pct']:>6.2f}%)")

    # Show sample trades with P&L
    print("\n" + "="*80)
    print("SAMPLE TRADES WITH P&L ATTRIBUTION (First 10 matured trades)")
    print("="*80)
    
    for i, trade in enumerate(matured_trades[:10]):
        pnl = trade.calculate_pnl_attribution()
        print(f"\n{i+1}. {trade.trade_id} | {trade.foreign_currency} | {trade.trade_date.strftime('%Y-%m-%d')}")
        print(f"   Direction: {trade.direction} | Notional: ${trade.trade_notional_base:,.0f}")
        print(f"   Carry P&L: ${pnl.carry_pnl:>10,.0f} | Spot P&L: ${pnl.spot_movement_pnl:>10,.0f} | "
              f"Total: ${pnl.total_pnl:>10,.0f}")
else:
    print(f"\n⚠️  No matured trades found with current data.")

ANALYZING MATURED TRADES

Trade Analysis as of: 2025-04-01
Total trades generated: 63
Matured trades: 63
Trades still open: 0

P&L SUMMARY

Aggregated P&L (All 63 matured trades):
  Total Notional:         $  15,398,140
  Total Carry P&L:        $      60,191  (  0.39%)
  Total Spot Movement:    $      37,969  (  0.25%)
  Total P&L:              $      98,160  (  0.64%)

SAMPLE TRADES WITH P&L ATTRIBUTION (First 10 matured trades)

1. TRADE-20200131-EUR | EUR | 2020-01-31
   Direction: SELL | Notional: $235,192
   Carry P&L: $     1,434 | Spot P&L: $     1,458 | Total: $     2,892

2. TRADE-20200131-GBP | GBP | 2020-01-31
   Direction: SELL | Notional: $45,422
   Carry P&L: $       143 | Spot P&L: $     3,570 | Total: $     3,713

3. TRADE-20200131-JPY | JPY | 2020-01-31
   Direction: BUY | Notional: $0
   Carry P&L: $         0 | Spot P&L: $        -0 | Total: $         0

4. TRADE-20200331-EUR | EUR | 2020-03-31
   Direction: SELL | Notional: $253,491
   Carry P&L: $       976 | Spot

### 3.1 Formatted P&L Attribution Table

Use the built-in formatter for clean, readable output.

In [9]:
# Use the new TradePortfolio reporting API
print(trade_portfolio.format_full_report())


TRADE PORTFOLIO SUMMARY
As of: 2025-04-01
Tenor: 3M
Currency: USD

Matured Trades: 63
Total Notional: USD 15,398,140

Carry P&L:       USD       60,191  (  0.39%)
Spot Movement:   USD       37,969  (  0.25%)
Total P&L:       USD       98,160  (  0.64%)



BY CURRENCY BREAKDOWN (in USD)
CCY    Trades        Notional       Carry P&L        Spot P&L       Total P&L
-----------------------------------------------------------------------------------------------
EUR        21 USD    11,036,776 USD        44,180 USD         7,348 USD        51,528
GBP        21 USD     3,008,650 USD         1,640 USD         1,569 USD         3,210
JPY        21 USD     1,352,714 USD        14,372 USD        29,051 USD        43,422


DETAILED TRADES (63 of 63) - Values in USD
Date         CCY    Dir    Spot@Inc    Forward   Spot@Mat     Notional    Carry PnL     Spot PnL    Total PnL
------------------------------------------------------------------------------------------------------------------------
2020

## 4. Single Trade P&L Examples

### 4.1 Standard Pair Example (EURUSD)

Demonstrate P&L calculation for a EURUSD hedge (standard quotation: foreign/base).

### 4.2 Inverted Pair Example (USDJPY)

Demonstrate P&L calculation for USDJPY (inverted quotation: base/foreign).

**Key Difference:**
- To hedge JPY exposure, we BUY USDJPY (equivalent to selling JPY, buying USD)
- Trade direction is opposite of standard pairs

## 5. Aggregate P&L by Currency

Analyze total P&L attribution by currency across all trades.

In [ ]:
# Get P&L aggregation by currency from TradePortfolio
# NEW API: by_currency property provides pre-aggregated metrics
by_currency = trade_portfolio.by_currency

print("="*80)
print("P&L ATTRIBUTION BY CURRENCY")
print("="*80)
print(f"\n{'Currency':<10} {'Trades':>8} {'Total Notional':>15} {'Carry P&L':>15} {'Spot P&L':>15} {'Total P&L':>15}")
print("-"*85)

total_notional = 0
total_carry = 0
total_spot = 0
total_pnl = 0

for currency in sorted(by_currency.keys()):
    data = by_currency[currency]
    print(f"{currency:<10} {data['trade_count']:>8} ${data['total_notional']:>14,.0f} "
          f"${data['carry_pnl']:>14,.0f} ${data['spot_pnl']:>14,.0f} ${data['total_pnl']:>14,.0f}")
    
    total_notional += data['total_notional']
    total_carry += data['carry_pnl']
    total_spot += data['spot_pnl']
    total_pnl += data['total_pnl']

print("-"*85)
print(f"{'TOTAL':<10} {len(trade_portfolio.matured_trades):>8} ${total_notional:>14,.0f} "
      f"${total_carry:>14,.0f} ${total_spot:>14,.0f} ${total_pnl:>14,.0f}")

# Show percentages
print("\n" + "="*80)
print("P&L ATTRIBUTION BY CURRENCY (% of Notional)")
print("="*80)
print(f"\n{'Currency':<10} {'Carry %':>12} {'Spot %':>12} {'Total %':>12}")
print("-"*50)

for currency in sorted(by_currency.keys()):
    data = by_currency[currency]
    print(f"{currency:<10} {data['carry_pnl_pct']:>11.2f}% {data['spot_pnl_pct']:>11.2f}% "
          f"{data['total_pnl_pct']:>11.2f}%")

# Display as DataFrame for easy inspection
print("\n" + "="*80)
print("DETAILED BREAKDOWN (DataFrame)")
print("="*80)

# Convert to DataFrame for display
df_by_ccy = pd.DataFrame(by_currency).T
display(df_by_ccy)

P&L ATTRIBUTION BY CURRENCY

Currency     Trades  Total Notional       Carry P&L        Spot P&L       Total P&L
-------------------------------------------------------------------------------------
EUR              21 $    11,036,776 $        44,180 $         7,348 $        51,528
GBP              21 $     3,008,650 $         1,640 $         1,569 $         3,210
JPY              21 $     1,352,714 $        14,372 $        29,051 $        43,422
-------------------------------------------------------------------------------------
TOTAL            63 $    15,398,140 $        60,191 $        37,969 $        98,160

P&L ATTRIBUTION BY CURRENCY (% of Notional)

Currency        Carry %       Spot %      Total %
--------------------------------------------------
EUR               0.40%        0.07%        0.47%
GBP               0.05%        0.05%        0.11%
JPY               1.06%        2.15%        3.21%

DETAILED BREAKDOWN (DataFrame)


,trade_count,total_notional,carry_pnl,spot_pnl,total_pnl,carry_pnl_pct,spot_pnl_pct,total_pnl_pct
EUR,21.0000,11036776.1025,44179.5761,7348.1448,51527.7209,0.4003,0.0666,0.4669
GBP,21.0000,3008649.7715,1640.1006,1569.4634,3209.5640,0.0545,0.0522,0.1067
JPY,21.0000,1352714.3084,14371.5112,29050.9705,43422.4817,1.0624,2.1476,3.2100


## 6. Visualization

### 6.1 Waterfall Chart - P&L Components by Currency

Visualize how carry and spot movement contribute to total P&L for each currency.

In [13]:
# Create waterfall chart using Plotly
fig = go.Figure()

# Prepare data for waterfall
currencies = sorted(by_currency.keys())
x_labels = []
y_values = []
measures = []

cumulative = 0
for currency in currencies:
    carry = by_currency[currency]['carry_pnl']
    spot = by_currency[currency]['spot_pnl']
    
    # Carry component
    x_labels.append(f"{currency} Carry")
    y_values.append(carry)
    measures.append('relative')
    
    # Spot component
    x_labels.append(f"{currency} Spot")
    y_values.append(spot)
    measures.append('relative')

# Total
total_pnl = sum(by_currency[c]['total_pnl'] for c in currencies)
x_labels.append('Total P&L')
y_values.append(total_pnl)
measures.append('total')

fig = go.Figure(go.Waterfall(
    x=x_labels,
    y=y_values,
    measure=measures,
    text=[f"${v:,.0f}" for v in y_values],
    textposition="outside",
    connector={"line": {"color": "rgb(63, 63, 63)"}},
    increasing={"marker": {"color": "green"}},
    decreasing={"marker": {"color": "red"}},
    totals={"marker": {"color": "blue"}}
))

fig.update_layout(
    title="P&L Attribution Waterfall: Carry vs Spot Movement by Currency",
    xaxis_title="Component",
    yaxis_title="P&L (USD)",
    height=600,
    showlegend=False
)

fig.show()

### 6.2 Stacked Bar Chart - P&L Components by Currency

### 6.3 Time Series - Carry vs Spot P&L Over Time

Visualize P&L components over time to understand patterns and trends.

### 6.4 P&L Decomposition by Currency Over Time

## 7. Use Cases & Interpretation

### 7.1 Performance Reporting

**Question:** How much did hedging cost us vs how much did we save from spot movements?

### 7.2 Hedge Effectiveness Analysis

**Question:** Did the hedges work as intended? Compare hedged vs unhedged scenarios.

### 7.3 Carry Cost Analysis

**Question:** Which currencies were expensive/cheap to hedge?

### 7.4 Strategy Comparison: What if we hedged differently?

Compare actual P&L attribution with alternative hedge ratios.

## 8. NEW: Time Series P&L Visualization (v2.7.0+)

Generate periodic time series analysis and visualizations of P&L components by currency.

This section demonstrates the new `TradePortfolio` methods for analyzing P&L decomposition over time with flexible frequency periods and cumulative analysis options.

**Available Methods:**
- `generate_pnl_timeseries()` - Generate time series data (periodic or cumulative)
- `plot_pnl_by_currency()` - Multi-subplot time series (Carry | Spot | Total)
- `plot_pnl_components()` - Stacked/grouped bar chart (Carry vs Spot)
- `plot_pnl_heatmap()` - Matrix visualization (Currencies × Periods)
- `plot_pnl_summary_dashboard()` - Comprehensive 4-panel dashboard

### 8.1 Generate Time Series Data (Periodic and Cumulative)

Generate P&L time series grouped by monthly periods with optional cumulative sums.

In [15]:
# 8.1.1 Generate PERIODIC (non-cumulative) time series grouped by QUARTER
print("="*80)
print("TIME SERIES DATA: QUARTERLY (Periodic)")
print("="*80)

ts_quarterly = trade_portfolio.generate_pnl_timeseries(period='Q', cumulative=False)
print(f"\nGenerated {len(ts_quarterly)} quarterly data points")
print(f"\nSample data (first 10 rows):")
display(ts_quarterly.head(10))

print("\n" + "="*80)
print("TIME SERIES DATA: QUARTERLY (Cumulative)")
print("="*80)

ts_quarterly_cumul = trade_portfolio.generate_pnl_timeseries(period='Q', cumulative=True)
print(f"\nCumulative quarterly P&L by currency")
print(f"Sample data (first 10 rows):")
display(ts_quarterly_cumul.head(10))

TIME SERIES DATA: QUARTERLY (Periodic)

Generated 54 quarterly data points

Sample data (first 10 rows):


/Volumes/ext1/fx_hedging_linear/fx_hedging_linear/src/hedge_trade.py:686: FutureWarning:

'Q' is deprecated and will be removed in a future version, please use 'QE' instead.



,date,currency,carry_pnl,spot_pnl,total_pnl
0,2020-03-31,EUR,2409.9075,-3687.6673,-1277.7598
1,2020-03-31,GBP,243.9949,3672.0000,3915.9949
2,2020-03-31,JPY,0.0000,0.0000,0.0000
3,2020-06-30,EUR,621.5429,-13194.9162,-12573.3733
4,2020-06-30,GBP,34.3952,-2737.8999,-2703.5047
5,2020-06-30,JPY,0.0000,0.0000,0.0000
6,2020-09-30,EUR,742.9312,-13908.8856,-13165.9544
7,2020-09-30,GBP,50.0787,-4109.3035,-4059.2248
8,2020-09-30,JPY,0.0000,0.0000,0.0000
9,2020-12-31,EUR,827.1584,15952.3412,16779.4996



TIME SERIES DATA: QUARTERLY (Cumulative)

Cumulative quarterly P&L by currency
Sample data (first 10 rows):


/Volumes/ext1/fx_hedging_linear/fx_hedging_linear/src/hedge_trade.py:686: FutureWarning:

'Q' is deprecated and will be removed in a future version, please use 'QE' instead.



,date,currency,carry_pnl,spot_pnl,total_pnl
0,2020-03-31,EUR,2409.9075,-3687.6673,-1277.7598
1,2020-03-31,GBP,243.9949,3672.0000,3915.9949
2,2020-03-31,JPY,0.0000,0.0000,0.0000
3,2020-06-30,EUR,3031.4503,-16882.5835,-13851.1331
4,2020-06-30,GBP,278.3901,934.1001,1212.4902
5,2020-06-30,JPY,0.0000,0.0000,0.0000
6,2020-09-30,EUR,3774.3815,-30791.4690,-27017.0875
7,2020-09-30,GBP,328.4688,-3175.2034,-2846.7346
8,2020-09-30,JPY,0.0000,0.0000,0.0000
9,2020-12-31,EUR,4601.5400,-14839.1279,-10237.5879


### 8.2 Plot Time Series by Currency (3-Panel: Carry | Spot | Total)

In [16]:
# 8.2.1 Quarterly P&L by currency (periodic)
print("\nPlotting quarterly P&L by currency (periodic)...")
fig1 = trade_portfolio.plot_pnl_by_currency(period='Q', cumulative=False)
fig1.show()

print("\n" + "="*80)

# 8.2.2 Quarterly P&L by currency (cumulative)
print("\nPlotting quarterly P&L by currency (cumulative)...")
fig2 = trade_portfolio.plot_pnl_by_currency(period='Q', cumulative=True)
fig2.show()

print("\n✓ Each subplot shows separate P&L component (Carry | Spot | Total)")
print("  One line per currency, easy to compare trends across currencies")


Plotting quarterly P&L by currency (periodic)...


/Volumes/ext1/fx_hedging_linear/fx_hedging_linear/src/hedge_trade.py:686: FutureWarning:

'Q' is deprecated and will be removed in a future version, please use 'QE' instead.





Plotting quarterly P&L by currency (cumulative)...


/Volumes/ext1/fx_hedging_linear/fx_hedging_linear/src/hedge_trade.py:686: FutureWarning:

'Q' is deprecated and will be removed in a future version, please use 'QE' instead.




✓ Each subplot shows separate P&L component (Carry | Spot | Total)
  One line per currency, easy to compare trends across currencies


### 8.3 Plot P&L Components (Carry vs Spot as Stacked Bars)

In [17]:
# 8.3.1 Aggregated carry vs spot across ALL currencies
print("\nPlotting aggregated P&L components (Carry vs Spot)...")
fig3 = trade_portfolio.plot_pnl_components(period='Q', cumulative=False, stacked=True)
fig3.show()

print("\n" + "="*80)

# 8.3.2 P&L components for single currency
print("\nPlotting P&L components for EUR only...")
fig4 = trade_portfolio.plot_pnl_components(period='Q', cumulative=False, currency='EUR', stacked=True)
fig4.show()

print("\n✓ Green = Carry P&L (benefit from forward premium/discount)")
print("  Red   = Spot Movement P&L (profit/loss from FX rate changes)")
print("\nAnalysis:")
print("  - Height of green bars = how much you earned from hedging")
print("  - Height of red bars = profit/loss from actual spot movements")
print("  - Shows clearly which component drove total P&L")


Plotting aggregated P&L components (Carry vs Spot)...


/Volumes/ext1/fx_hedging_linear/fx_hedging_linear/src/hedge_trade.py:686: FutureWarning:

'Q' is deprecated and will be removed in a future version, please use 'QE' instead.





Plotting P&L components for EUR only...


/Volumes/ext1/fx_hedging_linear/fx_hedging_linear/src/hedge_trade.py:686: FutureWarning:

'Q' is deprecated and will be removed in a future version, please use 'QE' instead.




✓ Green = Carry P&L (benefit from forward premium/discount)
  Red   = Spot Movement P&L (profit/loss from FX rate changes)

Analysis:
  - Height of green bars = how much you earned from hedging
  - Height of red bars = profit/loss from actual spot movements
  - Shows clearly which component drove total P&L


### 8.4 Plot P&L Heatmap (Currency × Period Matrix)

In [18]:
# 8.4.1 Total P&L heatmap
print("\nPlotting P&L heatmap (Currencies × Periods)...")
fig5 = trade_portfolio.plot_pnl_heatmap(period='Q', component='total', cumulative=False)
fig5.show()

print("\n" + "="*80)

# 8.4.2 Carry P&L heatmap
print("\nPlotting Carry P&L heatmap...")
fig6 = trade_portfolio.plot_pnl_heatmap(period='Q', component='carry', cumulative=False)
fig6.show()

print("\n✓ Color intensity shows magnitude of P&L:")
print("  Green = Positive P&L (gains)")
print("  Red   = Negative P&L (losses)")
print("\nUse cases:")
print("  - Quick pattern identification (which currencies/periods performed best)")
print("  - Compare carry vs spot across full history")
print("  - Identify seasonal or cyclical patterns in P&L")


Plotting P&L heatmap (Currencies × Periods)...


/Volumes/ext1/fx_hedging_linear/fx_hedging_linear/src/hedge_trade.py:686: FutureWarning:

'Q' is deprecated and will be removed in a future version, please use 'QE' instead.





Plotting Carry P&L heatmap...


/Volumes/ext1/fx_hedging_linear/fx_hedging_linear/src/hedge_trade.py:686: FutureWarning:

'Q' is deprecated and will be removed in a future version, please use 'QE' instead.




✓ Color intensity shows magnitude of P&L:
  Green = Positive P&L (gains)
  Red   = Negative P&L (losses)

Use cases:
  - Quick pattern identification (which currencies/periods performed best)
  - Compare carry vs spot across full history
  - Identify seasonal or cyclical patterns in P&L


### 8.5 Comprehensive P&L Dashboard (4-Panel Overview)

In [19]:
# 8.5.1 Comprehensive 4-panel dashboard
print("\nGenerating comprehensive P&L dashboard...")
fig7 = trade_portfolio.plot_pnl_summary_dashboard(period='QE')
fig7.show()

print("\n" + "="*80)
print("DASHBOARD PANELS EXPLAINED:")
print("="*80)
print("\n1. TOP-LEFT: Total P&L Over Time")
print("   - Shows quarterly P&L for each currency")
print("   - One line per currency for easy comparison")
print("   - Helps identify best/worst performing periods and currencies")

print("\n2. TOP-RIGHT: Cumulative P&L")
print("   - Running total of P&L over time")
print("   - Shows wealth accumulation from hedging program")
print("   - Trend indicates if strategy is improving/deteriorating")

print("\n3. BOTTOM-LEFT: Carry vs Spot Components (Stacked)")
print("   - Green bars: Carry P&L (earned from forward premium/discount)")
print("   - Red bars: Spot movement P&L (from FX rate changes)")
print("   - Stacked view shows total contribution from each component")

print("\n4. BOTTOM-RIGHT: Whole Period P&L Summary by Component")
print("   - Summary table of P&L across entire period by currency")
print("   - Carry P&L column: Total earnings from forward premium/discount")
print("   - Spot P&L column: Total gains/losses from FX rate changes")
print("   - Total P&L column: Combined carry and spot results for each currency")


Generating comprehensive P&L dashboard...



DASHBOARD PANELS EXPLAINED:

1. TOP-LEFT: Total P&L Over Time
   - Shows quarterly P&L for each currency
   - One line per currency for easy comparison
   - Helps identify best/worst performing periods and currencies

2. TOP-RIGHT: Cumulative P&L
   - Running total of P&L over time
   - Shows wealth accumulation from hedging program
   - Trend indicates if strategy is improving/deteriorating

3. BOTTOM-LEFT: Carry vs Spot Components (Stacked)
   - Green bars: Carry P&L (earned from forward premium/discount)
   - Red bars: Spot movement P&L (from FX rate changes)
   - Stacked view shows total contribution from each component

4. BOTTOM-RIGHT: Whole Period P&L Summary by Component
   - Summary table of P&L across entire period by currency
   - Carry P&L column: Total earnings from forward premium/discount
   - Spot P&L column: Total gains/losses from FX rate changes
   - Total P&L column: Combined carry and spot results for each currency


In [1]:
# Export notebook to HTML
!jupyter nbconvert \
    "04_pnl_attribution_demo.ipynb" \
    --to html \
    --template lab \
    --output "04_pnl_attribution_demo" \
    --output-dir output \
    --no-prompt

[NbConvertApp] Converting notebook 04_pnl_attribution_demo.ipynb to html
/Volumes/ext1/fx_hedging_linear/fx_hedging_linear/.venv/share/jupyter/nbconvert/templates/base/display_priority.j2:32: UserWarning: Your element with mimetype(s) dict_keys(['application/vnd.plotly.v1+json']) is not able to be represented.
  {%- elif type == 'text/vnd.mermaid' -%}
[NbConvertApp] Writing 406005 bytes to output/04_pnl_attribution_demo.html
